Code framework taken from HW file.

In [1]:
import random
def place_n_queens(n):
    assert n >= 0
    columns = [random.randrange(0,n) for x in range(n)]
    return columns


def displayBoard(columns):
    if not columns:
        return
    n = len(columns)
    print(columns)
    for rowVal in range(n):
        rowStr = ["." for x in range(n)]
        rowStr[columns[rowVal]] = '♛'
        print("  ".join(rowStr))
    print()


In [2]:
columns = place_n_queens(8)
displayBoard(columns)

[0, 2, 0, 1, 1, 7, 3, 5]
♛  .  .  .  .  .  .  .
.  .  ♛  .  .  .  .  .
♛  .  .  .  .  .  .  .
.  ♛  .  .  .  .  .  .
.  ♛  .  .  .  .  .  .
.  .  .  .  .  .  .  ♛
.  .  .  ♛  .  .  .  .
.  .  .  .  .  ♛  .  .



In [3]:
def place_in_next_row(col, columns):
    columns.append(col)

def remove_from_current_row(columns):
    if len(columns) > 0:
        return columns.pop()
    return -1

# better version

def is_place_in_next_row_valid(col, columns):
    # new row
    row = len(columns)

    # for each row with existing queen check if the current (row, col) position is not valid

    for queen_row, queen_column in enumerate(columns):
        if col == queen_column:
            return False
          # check if col in new row is in the same diagonal as existing queen
        if queen_column - queen_row == col - row:
            return False
          # check if col in new row is in the same anti-diagonal as existing queen
        if queen_column + queen_row == col + row:
            return False

    return True


Code from the given file.

In [4]:
def solve_queens_dfs(size):
    columns = []
    number_of_moves = 0
    number_of_iterations = 0
    row = 0 # current row
    col = 0 # always start each row at leftmost col
    # iterate over rows of board
    while True:
        #place queen in next row
        # print("I am trying to put the queen in row ", row)
        # print(columns)
        # print(f"number_of_moves: {number_of_moves}")
        while col < size:
            number_of_iterations += 1
            if is_place_in_next_row_valid(col, columns):
                place_in_next_row(col, columns)
                number_of_moves+=1
                # print(f"placed: row: {row}, col: {col}")
                row += 1
                col = 0
                break
            else:
                # print(f"not placed: row: {row}, col: {col}")
                col += 1

        # could not find an open col in this row,  or board is full
        if (col == size or row == size):
            number_of_iterations+=1
            # if board is full, we have a solution
            if row == size:
                #print("I did it! Here is my solution")
                #display(columns)
                converged = True
                return columns, number_of_iterations, number_of_moves, converged
            # else couldn't find a solution so need to backtrack
            #print("start to backtrack ... ")
            prev_col = remove_from_current_row(columns)
            number_of_moves+=1
            if (prev_col == -1): # backtracked past column 1
                print("There are no solutions")
                #print(number_of_moves)
                converged = False
                return columns, number_of_iterations, number_of_moves, converged
            # retry previous row again
            row -= 1
            # start to now check at col = (1 + value of prev_column in the row)
            col = 1 + prev_col



In [5]:
columns, num_iterations, number_moves, converged=solve_queens_dfs(8)
print(f"number of iterations: {num_iterations}")
print(f"number of moves: {number_moves}")
displayBoard(columns)

number of iterations: 982
number of moves: 218
[0, 4, 7, 5, 2, 6, 1, 3]
♛  .  .  .  .  .  .  .
.  .  .  .  ♛  .  .  .
.  .  .  .  .  .  .  ♛
.  .  .  .  .  ♛  .  .
.  .  ♛  .  .  .  .  .
.  .  .  .  .  .  ♛  .
.  ♛  .  .  .  .  .  .
.  .  .  ♛  .  .  .  .



Validity checker

In [6]:
def is_valid(columns):
    for row, col in enumerate(columns):
        for row2, col2 in enumerate(columns[row+1:], row+1):
            # Check if 2 queens are in the same column
            if col == col2:
                return False
            # Check if 2 queens are on the same diagonal
            if abs(col - col2) == abs(row - row2):
                return False
            # No need to check if same row since in this input format there can only be one per row
    return True

# Part one:

In [7]:
# The way this is implemented there can only be one queen per row (16 mil instead of 4 bil) so it is much faster then choosing 8 queens (it is still slow)
def british_museum(size):
    count = 1
    columns = place_n_queens(size)
    while(not is_valid(columns)):
        count +=1
        columns = place_n_queens(size)
    return columns, count, count*size, True # the number of moves is going to be each queen put down for each try so count*size

In [8]:
output = british_museum(8)
print(output)
displayBoard(output[0])

([6, 2, 0, 5, 7, 4, 1, 3], 67721, 541768, True)
[6, 2, 0, 5, 7, 4, 1, 3]
.  .  .  .  .  .  ♛  .
.  .  ♛  .  .  .  .  .
♛  .  .  .  .  .  .  .
.  .  .  .  .  ♛  .  .
.  .  .  .  .  .  .  ♛
.  .  .  .  ♛  .  .  .
.  ♛  .  .  .  .  .  .
.  .  .  ♛  .  .  .  .



# Part 2

The code for part 2 was given in solve_queens_dfs above already

In [9]:
output = solve_queens_dfs(25)
print(output)

([0, 2, 4, 1, 3, 8, 10, 12, 14, 18, 20, 23, 19, 24, 22, 5, 7, 9, 6, 13, 15, 17, 11, 16, 21], 1265434, 97341, True)


# Part 3 Heuristic Repair/Stochastic Search

Also the hill climing

In [10]:
amount = 1000

def get_constraints(columns):
    output = [0]*len(columns)
    for row, col in enumerate(columns):
        for row2, col2 in enumerate(columns[row+1:], row+1):
            # Check if 2 queens are in the same column
            if col == col2:
                output[row]+=1
                output[row2]+=1
            # Check if 2 queens are on the same diagonal
            if abs(col - col2) == abs(row - row2):
                output[row]+=1
                output[row2]+=1
    return output

# This gets the constraints for a single queen at a given position
def get_constraints_col(columns, row_val, col_val):
    output = 0
    for row, col in enumerate(columns):
        # If in current row, skip
        if(row == row_val):
            continue    
        # Check if 2 queens are in the same column
        if col == col_val:
            output+=1
        # Check if 2 queens are on the same diagonal
        if abs(col - col_val) == abs(row - row_val):
            output+=1
    return output

###     Part A Basic Hill Climbing

In [11]:
def basic_hill_climbing(size):
    columns = place_n_queens(size)
    count = 1 # Starts at 1 since we generated a random board
    moves = 0
    while(not is_valid(columns) and count<amount):
        con = get_constraints(columns)
        pos = con.index(max(con))
        val = con[pos]
        low = columns[pos]
        for i in range(len(columns)):
            con_temp = get_constraints_col(columns, pos, i)
            if (con_temp<val):
                val = con_temp
                low = i
            moves +=1 # Each time this runs we move the queen into a new position to see if that pos has less constraints
        columns[pos] = low
        count+=1 # This is the amount of times that we tested a new board state
    return columns, count, moves, is_valid(columns)

In [12]:
output = basic_hill_climbing(4)
print(output)
displayBoard(output[0])

([1, 3, 0, 0], 1000, 3996, False)
[1, 3, 0, 0]
.  ♛  .  .
.  .  .  ♛
♛  .  .  .
♛  .  .  .



### Part B slightly more advanced hill climbing

In [13]:
def slightly_more_advanced_hill_climbing(size): # Same as basic, but we are adding a restart if it gets stuck or wanders too long.
    columns = place_n_queens(size)
    count = 1 # Starts at 1 since we generated a random board
    moves = 0
    sub_count = 0
    while(not is_valid(columns) and count<amount):
        if(sub_count>=2*size): # Chose this number at random and it worked well
            columns = place_n_queens(size)
            sub_count = 0
            continue
        con = get_constraints(columns)
        pos = con.index(max(con))
        val = con[pos]
        low = columns[pos]
        for i in range(len(columns)):
            con_temp = get_constraints_col(columns, pos, i)
            if (con_temp<val):
                val = con_temp
                low = i
            moves +=1 # Each time this runs we move the queen into a new position to see if that pos has less constraints
        if(low == columns[pos]): # If there is no better move restart
            columns = place_n_queens(size)
            sub_count = 0
            count += 1
            continue
        columns[pos] = low
        count+=1 # This is the amount of times that we tested a new board state
        sub_count +=1
    return columns, count, moves, is_valid(columns)

In [14]:
output = slightly_more_advanced_hill_climbing(32)
print(output)

([9, 13, 17, 8, 1, 28, 16, 23, 15, 10, 12, 4, 31, 11, 20, 5, 30, 12, 7, 19, 24, 6, 15, 3, 18, 21, 14, 14, 0, 2, 25, 1], 1000, 31968, False)


### Part C sideways correction

In [15]:
def sideways_correction (size): # same as before but will try going to equivalent value sideways if it cannot find a better location
    columns = place_n_queens(size)
    count = 1 # Starts at 1 since we generated a random board
    moves = 0
    sub_count = 0
    while(count<amount):
        if(sub_count>=2*size): # Chose this number at random and it worked well
            columns = place_n_queens(size)
            sub_count = 0
            continue
        con = get_constraints(columns)
        if(max(con) == 0):
            break

        # Randomly choose one of the most constrained queens (while debugging, I found that this is faster)
        max_con = max(con)
        positions = [i for i, x in enumerate(con) if x == max_con]
        pos = random.choice(positions)

        val = con[pos]
        low = columns[pos]
        others = []
        for i in range(len(columns)):
            con_temp = get_constraints_col(columns, pos, i)
            if (con_temp<val):
                val = con_temp
                low = i
            elif(con_temp == val and i != columns[pos]):
                others.append(i)
            moves +=1 # Each time this runs we move the queen into a new position to see if that pos has less constraints
        if low == columns[pos]: # If no better move try to go sideways, if cannot go sideways then restart
            if others:
                low = random.choice(others)
            else:
                columns = place_n_queens(size)
                sub_count = 0
                continue
        columns[pos] = low
        count+=1 # This is the amount of times that we tested a new board state
        sub_count +=1
    return columns, count, moves, is_valid(columns)

In [16]:
output = sideways_correction(500)
print(output[-1])
print(output)

True
([294, 325, 74, 310, 441, 258, 188, 169, 494, 190, 239, 357, 273, 166, 297, 319, 281, 105, 492, 376, 496, 73, 122, 246, 236, 240, 141, 232, 476, 444, 452, 153, 114, 318, 350, 277, 293, 404, 410, 130, 394, 307, 361, 49, 181, 161, 351, 367, 282, 414, 332, 1, 223, 54, 256, 36, 50, 481, 133, 340, 493, 16, 159, 19, 216, 412, 52, 199, 298, 424, 495, 429, 393, 39, 348, 381, 463, 184, 468, 58, 390, 145, 290, 26, 359, 485, 266, 40, 164, 202, 458, 278, 121, 299, 135, 445, 224, 24, 179, 63, 243, 155, 196, 253, 27, 497, 32, 138, 90, 421, 447, 355, 211, 140, 475, 30, 334, 151, 11, 378, 168, 182, 370, 41, 183, 47, 342, 284, 287, 259, 407, 77, 108, 354, 194, 69, 81, 18, 22, 137, 53, 221, 219, 64, 280, 431, 272, 167, 33, 109, 97, 154, 207, 426, 42, 185, 198, 48, 330, 89, 453, 262, 289, 415, 365, 136, 483, 380, 242, 397, 439, 14, 267, 101, 362, 28, 117, 417, 201, 488, 477, 435, 438, 84, 296, 204, 356, 94, 413, 152, 336, 405, 10, 314, 464, 127, 106, 193, 474, 132, 46, 321, 312, 252, 382, 249, 484, 

### Part D Any Improvement

In [17]:
def any_improvment (size): # Same as before, but will take the first improvement.
    columns = place_n_queens(size)
    count = 1 # Starts at 1 since we generated a random board
    moves = 0
    sub_count = 0
    while(count<amount):
        if(sub_count>=2*size):
            columns = place_n_queens(size)
            sub_count = 0
            continue
        con = get_constraints(columns)
        if(max(con) == 0):
            break

        # Randomly choose one of the most constrained queens (while debugging, I found that this is faster)
        max_con = max(con)
        positions = [i for i, x in enumerate(con) if x == max_con]
        pos = random.choice(positions)

        val = con[pos]
        low = columns[pos]
        others = []
        for i in range(len(columns)):
            con_temp = get_constraints_col(columns, pos, i)
            if (con_temp<val):
                val = con_temp
                low = i
                break
            elif(con_temp == val and i != columns[pos]):
                others.append(i)
            moves +=1 # Each time this runs we move the queen into a new position to see if that pos has less constraints
        if low == columns[pos]: # If no better move try to go sideways, if cannot go sideways then restart
            if others:
                low = random.choice(others)
            else:
                columns = place_n_queens(size)
                sub_count = 0
                continue
        columns[pos] = low
        count+=1 # This is the amount of times that we tested a new board state
        sub_count +=1
    return columns, count, moves, is_valid(columns)

In [18]:
output = any_improvment(100)
print(output[-1])
print(output)

True
([42, 40, 90, 56, 35, 57, 22, 81, 89, 80, 64, 83, 99, 4, 92, 21, 48, 43, 98, 62, 10, 72, 3, 9, 37, 65, 25, 50, 1, 29, 15, 93, 68, 38, 6, 31, 85, 70, 30, 34, 58, 44, 13, 32, 28, 11, 33, 76, 36, 53, 69, 2, 66, 78, 95, 23, 86, 26, 59, 79, 87, 73, 20, 7, 16, 74, 60, 24, 96, 0, 67, 41, 82, 47, 39, 12, 97, 94, 27, 77, 63, 14, 84, 91, 19, 61, 49, 54, 8, 51, 46, 17, 71, 75, 5, 88, 55, 18, 52, 45], 184, 10824, True)


# Part 4 Forward checking 

Also part E

After a lot of experimentation, I determined that the most efficient version is one that chooses the row with the fewest available spaces next and places the queen randomly in an available spot. (There were also bugs in my code, so I rewrote it from scratch and made these changes while rewriting it.)

In [19]:
def search(board, available, number_of_iterations, number_of_moves):
    index = min((i for i in range(len(available)) if len(available[i]) > 0), key=lambda i: len(available[i]))
    available_new = [s.copy() for s in available]
    cur = list(available_new[index])
    while True:
        if not cur:
            return board, number_of_iterations, number_of_moves, False
        loc = random.choice(cur)
        number_of_iterations += 1
        cur.remove(loc)
        available_new[index] = set()
        number_of_moves += 1
        for i in range(len(available_new)):
            if i == index:
                continue            
            distance = abs(i - index)
            available_new[i].discard(loc)             # Same column
            available_new[i].discard(loc + distance)  # Right diagonal
            available_new[i].discard(loc - distance)  # Left diagonal
        board[index] = loc

        if(not all(j for i, j in enumerate(available_new) if board[i] == -1)): # This is the forward checking
            available_new = [s.copy() for s in available]
            board[index] = -1
            number_of_moves += 1
            continue

        if(any(available_new)):
            out_board,  number_of_iterations, number_of_moves, sucsess = search(board, available_new, number_of_iterations, number_of_moves)
            if(sucsess):
                return out_board,  number_of_iterations, number_of_moves, sucsess
            else:
                available_new = [s.copy() for s in available]
                board[index] = -1
                number_of_moves += 1
                continue
        else:
            if(any(i == -1 for i in board)):
                available_new = [s.copy() for s in available]
                board[index] = -1
                number_of_moves += 1
                continue
            else:
                return board, number_of_iterations, number_of_moves, True

def forward_checking(size):
    board = [-1]*size
    available = [set(range(size)) for _ in range(size)]
    number_of_iterations = 0
    number_of_moves = 0
    return search(board, available, number_of_iterations, number_of_moves)



In [31]:
output = forward_checking(100)
print(is_valid(output[0]))
print(output)

True
([58, 54, 0, 18, 66, 22, 1, 97, 86, 51, 65, 48, 94, 85, 75, 8, 30, 21, 34, 67, 16, 41, 12, 42, 2, 89, 7, 56, 36, 32, 52, 49, 95, 24, 14, 80, 15, 76, 92, 40, 78, 87, 26, 69, 9, 47, 45, 99, 71, 62, 38, 25, 29, 84, 17, 79, 13, 93, 64, 28, 43, 91, 33, 88, 20, 70, 27, 74, 55, 3, 37, 39, 81, 4, 59, 57, 23, 63, 19, 11, 77, 73, 44, 83, 96, 31, 61, 98, 60, 53, 35, 10, 68, 82, 6, 50, 90, 5, 46, 72], 1741, 3382, True)


Write a loop solving and printing each of the n-Queens problems from 10 to 100.  Try to get both of these approaches all the way to 100.  Did you need to add clever tricks to make it happen? Add a time counter for this loop as per the example in the homework handout.

In [25]:

import time
from IPython.display import display, Markdown

algorithms = [
    ("British Museum", british_museum, range(8, 9)),
    ("DFS with Backtracking", solve_queens_dfs, range(8, 26)),
    ("Basic Hill Climbing", basic_hill_climbing, range(8, 101)),
    ("Hill Climbing with Restarts", slightly_more_advanced_hill_climbing, range(8, 101)),
    ("Hill Climbing with Sideways Moves", sideways_correction, range(8, 101)),
    ("First-Choice Hill Climbing", any_improvment, range(8, 101)),
    ("Forward Checking", forward_checking, range(8, 101))
]

table = [
    "| Algorithm | Time (s) | Solved | Iterations | Moves |",
    "|---|---:|---:|---:|---:|"
]

for name, algorithm, sizes in algorithms:
    total_time = 0
    total_iterations = 0
    total_moves = 0
    solved = 0
    attempted = 0

    for n in sizes:
        start = time.perf_counter()
        board, iterations, moves, converged = algorithm(n)
        total_time += time.perf_counter() - start

        total_iterations += iterations
        total_moves += moves
        attempted += 1

        if (converged and len(board) == n and
            all(0 <= col < n for col in board) and is_valid(board)):
            solved += 1

    table.append(
        f"| {name} | {total_time:.4f} | {solved}/{attempted} | "
        f"{total_iterations:,} | {total_moves:,} |"
    )

markdown_table = "\n".join(table)
print(markdown_table)


| Algorithm | Time (s) | Solved | Iterations | Moves |
|---|---:|---:|---:|---:|
| British Museum | 0.9741 | 1/1 | 450,181 | 3,601,448 |
| DFS with Backtracking | 27.4680 | 18/18 | 57,660,976 | 4,988,327 |
| Basic Hill Climbing | 43.8107 | 0/93 | 93,000 | 5,016,978 |
| Hill Climbing with Restarts | 33.7086 | 7/93 | 88,140 | 4,964,957 |
| Hill Climbing with Sideways Moves | 3.5406 | 93/93 | 10,426 | 554,577 |
| First-Choice Hill Climbing | 3.4866 | 93/93 | 13,733 | 444,209 |
| Forward Checking | 0.8870 | 93/93 | 36,250 | 67,478 |


| Algorithm | Time (s) | Solved | Iterations | Moves |
|---|---:|---:|---:|---:|
| British Museum | 0.9741 | 1/1 | 450,181 | 3,601,448 |
| DFS with Backtracking | 27.4680 | 18/18 | 57,660,976 | 4,988,327 |
| Basic Hill Climbing | 43.8107 | 0/93 | 93,000 | 5,016,978 |
| Hill Climbing with Restarts | 33.7086 | 7/93 | 88,140 | 4,964,957 |
| Hill Climbing with Sideways Moves | 3.5406 | 93/93 | 10,426 | 554,577 |
| First-Choice Hill Climbing | 3.4866 | 93/93 | 13,733 | 444,209 |
| Forward Checking | 0.8870 | 93/93 | 36,250 | 67,478 |

# Reflection

This was an interesting assignment. The fact that basic hill climbing didn't really work isn't that surprising since you expect it to get stuck at local maximum, it also isn't so surprising that restarts didn't help because the issue is that there are many local maximums not that we started at a bad location. It is interesting though that if you allow sideways moves you actually can get the success rate up really high. It wasn't obvious to me at the outset that this would get around all of the common local maximums. It was also cool to see how a much simpler heuristic just using first choice actually ended up working better. Also I did a bunch of experimentations and if you add in some extra randomization for example randomize which column you choose first that substantially increases the speed and the success rate. This makes a lot of sense in retrospect since it increases the amount that you explore the possibility space and thereby makes it easier to not get stuck in a local maximum. The forward checking one was the most interesting, I added in the randomness but even without the randomness it performs roughly the same just maybe a little bit slower because it doesn't do the full search since it can guess right at the beginning. This actually explicitly checks to see if a move seems feasible using the forward checking heuristic and it is surprising that that heuristic is powerful enough to carry the algorithm through all the way to the end. But we do see with forward checking is that because of the randomization I added in since we choose a random column Sometimes it does take substantially longer. 